# 04 - Model Comparison

This notebook compares all volatility forecasting models using statistical tests.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from src.evaluation.metrics import evaluate_forecast, qlike, mse
from src.evaluation.statistical_tests import (
    diebold_mariano_test,
    model_confidence_set,
    mincer_zarnowitz_test,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Predictions

In [ ]:
# Load test predictions
PRED_DIR = Path("../data/predictions/test_2019")

# For demo, create synthetic predictions
np.random.seed(42)
n = 250  # ~1 year of trading days

dates = pd.date_range('2019-01-01', periods=n, freq='B')
y_true = np.abs(np.random.randn(n) * 0.001 + 0.0005)

# Simulated predictions from different models
predictions = pd.DataFrame({
    'date': dates,
    'y_true': y_true,
    'y_pred_har': y_true + np.random.randn(n) * 0.0002,
    'y_pred_garch': y_true + np.random.randn(n) * 0.00025,
    'y_pred_lightgbm': y_true + np.random.randn(n) * 0.00018,
    'y_pred_xgboost': y_true + np.random.randn(n) * 0.00019,
    'y_pred_gru': y_true + np.random.randn(n) * 0.00022,
    'y_pred_lstm': y_true + np.random.randn(n) * 0.00021,
    'y_pred_hybrid': y_true + np.random.randn(n) * 0.00017,
})

# Ensure positive predictions
for col in predictions.columns:
    if col.startswith('y_pred'):
        predictions[col] = np.abs(predictions[col]) + 1e-8

predictions.head()

## 2. Calculate Metrics for All Models

In [ ]:
# Calculate metrics
pred_cols = [c for c in predictions.columns if c.startswith('y_pred')]
model_names = [c.replace('y_pred_', '') for c in pred_cols]

results = []
for col, name in zip(pred_cols, model_names):
    metrics = evaluate_forecast(predictions['y_true'].values, predictions[col].values)
    results.append({
        'Model': name.upper(),
        'QLIKE': metrics.qlike,
        'MSE': metrics.mse,
        'RMSE': metrics.rmse,
        'MAE': metrics.mae,
        'R²': metrics.r2,
        'Corr': metrics.correlation,
    })

metrics_df = pd.DataFrame(results)
metrics_df['Rank'] = metrics_df['QLIKE'].rank().astype(int)
metrics_df = metrics_df.sort_values('QLIKE')
metrics_df

## 3. QLIKE Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.Set2(np.linspace(0, 1, len(metrics_df)))
bars = ax.barh(metrics_df['Model'], metrics_df['QLIKE'], color=colors)

# Add value labels
for bar, val in zip(bars, metrics_df['QLIKE']):
    ax.text(val + 0.0001, bar.get_y() + bar.get_height()/2,
           f'{val:.4f}', va='center', fontsize=10)

ax.set_xlabel('QLIKE (lower is better)')
ax.set_title('Model Comparison by QLIKE')

plt.tight_layout()
plt.show()

## 4. Diebold-Mariano Test

In [ ]:
# Pairwise DM tests
y_true = predictions['y_true'].values
dm_results = []

for i, (col1, name1) in enumerate(zip(pred_cols, model_names)):
    for j, (col2, name2) in enumerate(zip(pred_cols, model_names)):
        if i >= j:
            continue
            
        e1 = y_true - predictions[col1].values
        e2 = y_true - predictions[col2].values
        
        stat, pvalue = diebold_mariano_test(e1, e2, h=1)
        
        dm_results.append({
            'Model 1': name1.upper(),
            'Model 2': name2.upper(),
            'DM Stat': stat,
            'p-value': pvalue,
            'Significant': pvalue < 0.05,
        })

dm_df = pd.DataFrame(dm_results)
dm_df

## 5. Model Confidence Set

In [ ]:
# Calculate losses for MCS
losses = pd.DataFrame()
losses['date'] = predictions['date']

for col, name in zip(pred_cols, model_names):
    y_pred = predictions[col].values
    # QLIKE loss per observation
    losses[name] = y_true / y_pred - np.log(y_true / y_pred) - 1

losses_only = losses.drop('date', axis=1)

# Run MCS test
mcs_models = model_confidence_set(losses_only, alpha=0.1)
print(f"Models in MCS (alpha=0.1): {mcs_models}")

In [ ]:
# Add MCS membership to metrics table
metrics_df['In MCS'] = metrics_df['Model'].str.lower().isin(mcs_models)
metrics_df

## 6. Mincer-Zarnowitz Test

In [ ]:
# MZ test for each model
mz_results = []

for col, name in zip(pred_cols, model_names):
    y_pred = predictions[col].values
    
    result = mincer_zarnowitz_test(y_true, y_pred)
    
    mz_results.append({
        'Model': name.upper(),
        'Alpha': result['alpha'],
        'Beta': result['beta'],
        'R²': result['r2'],
        'F-stat': result['f_stat'],
        'p-value': result['p_value'],
        'Unbiased': result['p_value'] > 0.05,
    })

mz_df = pd.DataFrame(mz_results)
mz_df

## 7. Predictions Visualization

In [ ]:
# Plot top 3 models vs actual
top_models = metrics_df['Model'].head(3).str.lower().tolist()

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(predictions['date'], predictions['y_true'], 
       label='Actual', color='black', linewidth=1.5)

colors = ['blue', 'orange', 'green']
for model, color in zip(top_models, colors):
    ax.plot(predictions['date'], predictions[f'y_pred_{model}'],
           label=model.upper(), alpha=0.7, linewidth=1)

ax.set_xlabel('Date')
ax.set_ylabel('Realized Volatility')
ax.set_title('Top 3 Models: Actual vs Predicted')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Summary Table

In [ ]:
# Final summary
summary = metrics_df[['Model', 'QLIKE', 'RMSE', 'R²', 'Rank', 'In MCS']].copy()
print("\n" + "="*60)
print("VOLATILITY FORECASTING MODEL COMPARISON")
print("="*60)
print(f"\nTest Period: {predictions['date'].min().date()} to {predictions['date'].max().date()}")
print(f"Observations: {len(predictions)}")
print(f"\nModels in MCS (alpha=0.1): {mcs_models}")
print("\n" + summary.to_string(index=False))